# Prepoznavanje emocija iz govora (RAVDESS) — kompletan pipeline za Google Colab

Ova sveska sadrži **ceo, ispravljen pipeline** za diplomski rad: pripremu podataka, ekstrakciju karakteristika, treniranje i evaluaciju **KNN**, **Naive Bayes** i **1D CNN** modela, kao i **novog, četvrtog modela zasnovanog na Wav2Vec2** (transfer learning / fine-tuning pretreniranog transformera za govor), i na kraju poređenje svih modela.

**Ključna metodološka ispravka u odnosu na raniju verziju projekta:** podela na train/val/test je **po govorniku (actor-independent split)**, a ne nasumična stratifikovana podela. Ranija nasumična podela je dozvoljavala da se isti glumac (isti glas) nađe i u train i u test skupu, što veštački naduvava tačnost (data leakage) — model bi delimično prepoznavao *glas*, a ne *emociju*. Zbog toga su brojevi u ovoj svesci niži nego u starim rezultatima, ali su realniji i ispravniji za odbranu diplomskog rada.

**Sveska je podeljena u dva dela:**
- **Sekcije 1-9** — sve za KNN, Naive Bayes i CNN, sa finalnom tabelom/grafikom poređenja. Ovo pokreni sada (Runtime → Run all, ili ćeliju po ćeliju).
- **Sekcije 10-11** — Wav2Vec2 (novi, četvrti model, transfer learning) i ažurirano poređenje. Ovo možeš da uradiš kasnije, kad budeš spreman/na — samo ponovo pokreni ceo notebook od početka.

Trajanje (sekcije 1-9): KNN/NB feature ekstrakcija + augmentacija ~10 min, CNN feature ekstrakcija + augmentacija ~10 min, KNN grid search ~1 min, Naive Bayes ~10s, CNN trening par minuta (GPU) / desetak minuta (CPU). Ukupno, oko 25-35 minuta. Sekcija 10 (Wav2Vec2) dodatnih ~15-30 min na GPU-u.

**Pre pokretanja:** Runtime → Change runtime type → GPU (T4), da CNN i Wav2Vec2 trening budu brzi.


## 1. Setup — kloniranje repozitorijuma i instalacija paketa

In [ ]:
import os

REPO_URL = "https://github.com/andjelavostic/emotion-detector.git"
REPO_DIR = "emotion-detector"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}

%cd {REPO_DIR}

# Instaliramo pakete direktno (ne iz requirements.txt) da ova sveska radi
# i ako requirements.txt jos nije pushovan na GitHub.
!pip install -q "librosa==0.10.2.post1" soundfile resampy joblib


Ako `git clone` ne uspe (npr. repo je privatan), otpakuj svoj projekat na Google Drive i umesto gornje ćelije pokreni:

```python
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/putanja/do/emotion-detector
```


In [ ]:
import numpy as np
import pandas as pd
import librosa
import joblib
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

DATA_RAW = "data/raw"
DATA_PROC = "data/processed_data"
MODELS_DIR = "models"
EMOTIONS = ["angry", "calm", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

os.makedirs(f"{DATA_PROC}/knn_and_nb", exist_ok=True)
os.makedirs(f"{DATA_PROC}/cnn", exist_ok=True)
for m in ["knn", "naive_bayes", "cnn", "wav2vec2"]:
    os.makedirs(f"{MODELS_DIR}/{m}", exist_ok=True)

def save_confusion_matrix(y_true, y_pred, classes, filename, title):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(classes)))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(title)
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    plt.savefig(filename)
    plt.show()
    plt.close()


## 2. Podela po glumcu (actor-independent split)

RAVDESS ima 24 glumca. Izdvajamo 4 glumca za validaciju i 4 za test — **ti glumci se nikad ne pojavljuju u trening skupu.** Ovo je jedina metodološki ispravna podela za ovaj zadatak.

In [ ]:
TEST_ACTORS = [21, 22, 23, 24]
VAL_ACTORS  = [17, 18, 19, 20]
TRAIN_ACTORS = [a for a in range(1, 25) if a not in TEST_ACTORS + VAL_ACTORS]

print(f"Train glumci ({len(TRAIN_ACTORS)}): {TRAIN_ACTORS}")
print(f"Val glumci   ({len(VAL_ACTORS)}): {VAL_ACTORS}")
print(f"Test glumci  ({len(TEST_ACTORS)}): {TEST_ACTORS}")


## 3. Priprema labela iz imena fajlova

In [ ]:
emotion, gender, actor, path_list = [], [], [], []

for folder in sorted(os.listdir(DATA_RAW)):
    if not folder.startswith("Actor"):
        continue
    for file in os.listdir(os.path.join(DATA_RAW, folder)):
        if not file.endswith(".wav"):
            continue
        parts = file.split('-')
        emo_id = int(parts[2])
        actor_id = int(parts[6].split('.')[0])
        gen = "female" if actor_id % 2 == 0 else "male"

        emotion.append(emo_id)
        gender.append(gen)
        actor.append(actor_id)
        path_list.append(os.path.join(DATA_RAW, folder, file))

df_labels = pd.DataFrame({"emotion": emotion, "gender": gender, "actor": actor, "path": path_list})

emotion_map = {1: 'neutral', 2: 'calm', 3: 'happy', 4: 'sad', 5: 'angry', 6: 'fear', 7: 'disgust', 8: 'surprise'}
df_labels['emotion'] = df_labels['emotion'].map(emotion_map)

df_labels.to_csv(f"{DATA_PROC}/audio_labels.csv", index=False)
print("Sacuvani labeli:", df_labels.shape)
df_labels.head()


## 4. KNN / Naive Bayes — ekstrakcija Mel karakteristika

Za KNN i Naive Bayes se svaki audio isečak svodi na fiksni vektor: mean/std/max Mel-spektrograma (128 mel-frekvencija × 3 statistike = 384 karakteristike).

In [ ]:
def extract_mel_stats(path, duration=3, sr=44100, offset=0.5, n_mels=128):
    y, sr = librosa.load(path, duration=duration, sr=sr, offset=offset, res_type='kaiser_fast')
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, fmax=8000)
    mel_db = librosa.power_to_db(mel)
    mel_db = np.nan_to_num(mel_db, nan=0.0, posinf=0.0, neginf=0.0)
    mean_f = np.mean(mel_db, axis=1)
    std_f  = np.std(mel_db, axis=1)
    max_f  = np.max(mel_db, axis=1)
    return np.concatenate([mean_f, std_f, max_f])

features = []
for path in tqdm(df_labels['path'], desc="KNN/NB feature ekstrakcija"):
    features.append(extract_mel_stats(path))

feature_df = pd.DataFrame(features)
final_df = pd.concat([df_labels[['emotion', 'gender', 'actor', 'path']], feature_df], axis=1)
final_df.to_csv(f"{DATA_PROC}/knn_and_nb/features_mel_extended.csv", index=False)
print("Sacuvan feature dataframe:", final_df.shape)


### Podela na train/val/test (po glumcu)

In [ ]:
feat_df = pd.read_csv(f"{DATA_PROC}/knn_and_nb/features_mel_extended.csv")

train_df = feat_df[feat_df['actor'].isin(TRAIN_ACTORS)]
val_df   = feat_df[feat_df['actor'].isin(VAL_ACTORS)]
test_df  = feat_df[feat_df['actor'].isin(TEST_ACTORS)]

train_df.to_csv(f"{DATA_PROC}/knn_and_nb/features_mel_train.csv", index=False)
val_df.to_csv(f"{DATA_PROC}/knn_and_nb/features_mel_val.csv", index=False)
test_df.to_csv(f"{DATA_PROC}/knn_and_nb/features_mel_test.csv", index=False)

print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")


### Augmentacija (samo trening skup)

Augmentujemo **isključivo trening skup** (šum, pomeraj u vremenu, time-stretch, pitch-shift, promena jačine) — validacija i test ostaju netaknuti, jer treba da mere performanse na *realnim*, neizmenjenim snimcima.

In [ ]:
def add_noise(data, noise_factor=0.005):
    return data + noise_factor * np.random.randn(len(data))

def time_shift(data, shift_max=0.2):
    shift = np.random.randint(int(len(data) * -shift_max), int(len(data) * shift_max))
    return np.roll(data, shift)

def stretch(data, rate=None):
    if rate is None:
        rate = np.random.uniform(0.8, 1.2)
    return librosa.effects.time_stretch(data, rate=rate)

def pitch_shift(data, sr, n_steps=None):
    if n_steps is None:
        n_steps = np.random.uniform(-2, 2)
    return librosa.effects.pitch_shift(data, sr=sr, n_steps=n_steps)

def dynamic_change(data):
    return data * np.random.uniform(0.7, 1.3)

augment_funcs = [
    lambda y, sr: y,                 # original
    lambda y, sr: add_noise(y),
    lambda y, sr: time_shift(y),
    lambda y, sr: stretch(y),
    lambda y, sr: pitch_shift(y, sr),
    lambda y, sr: dynamic_change(y),
]

all_features, all_emotions, all_genders = [], [], []
sr = 44100

for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Augmentacija (KNN/NB train)"):
    y, _ = librosa.load(row['path'], duration=3, sr=sr, offset=0.5, res_type='kaiser_fast')
    for func in augment_funcs:
        y_aug = func(y, sr)
        mel = librosa.feature.melspectrogram(y=y_aug, sr=sr, n_mels=128, fmax=8000)
        mel_db = librosa.power_to_db(mel)
        mel_db = np.nan_to_num(mel_db, nan=0.0, posinf=0.0, neginf=0.0)
        feat = np.concatenate([np.mean(mel_db, axis=1), np.std(mel_db, axis=1), np.max(mel_db, axis=1)])
        all_features.append(feat)
        all_emotions.append(row['emotion'])
        all_genders.append(row['gender'])

df_aug_train = pd.DataFrame(all_features)
df_aug_train['emotion'] = all_emotions
df_aug_train['gender'] = all_genders
cols = ['emotion', 'gender'] + [c for c in df_aug_train.columns if c not in ['emotion', 'gender']]
df_aug_train = df_aug_train[cols]
df_aug_train.to_csv(f"{DATA_PROC}/knn_and_nb/features_mel_train_augmented.csv", index=False)
print("Augmented TRAIN dataset saved:", df_aug_train.shape)


### Enkodiranje labela i spajanje train+val

In [ ]:
train_aug = pd.read_csv(f"{DATA_PROC}/knn_and_nb/features_mel_train_augmented.csv")
val_feat  = pd.read_csv(f"{DATA_PROC}/knn_and_nb/features_mel_val.csv")
test_feat = pd.read_csv(f"{DATA_PROC}/knn_and_nb/features_mel_test.csv")

X_train = train_aug.drop(columns=['emotion', 'gender']).values
y_train = train_aug['emotion'].values

X_val   = val_feat.drop(columns=['emotion', 'gender', 'actor', 'path']).values
y_val   = val_feat['emotion'].values

X_test  = test_feat.drop(columns=['emotion', 'gender', 'actor', 'path']).values
y_test  = test_feat['emotion'].values

le_knn = LabelEncoder()
le_knn.fit(y_train)
y_test_enc = le_knn.transform(y_test)

# GridSearchCV ima SVOJU unutrasnju unakrsnu proveru (cv=5) da bi birao hiperparametre -
# ne treba mu posebno odvojen val skup za to. Zato train+val spajamo odmah ovde: to
# postaje "trening pool" iz kog GridSearchCV sam pravi svoje interne podele, a na kraju
# (podrazumevano, refit=True) automatski trenira odabrani najbolji model na CELOM tom
# pool-u. Test ostaje netaknut do same, finalne evaluacije.
X_trainval = np.concatenate([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])
y_trainval_enc = le_knn.transform(y_trainval)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Train+Val (koristi se za trening): {X_trainval.shape}, Test: {X_test.shape}")


## 5. Trening — KNN

`GridSearchCV` bira najbolje hiperparametre (`n_neighbors`, `weights`, `p`) koristeći svoju internu unakrsnu proveru nad **train+val** skupom, i automatski trenira finalni model na celom tom skupu (`refit=True`, podrazumevano ponašanje). Test skup se koristi **samo jednom, na kraju**, isključivo za merenje — nikad za biranje ni treniranje.

In [ ]:
scaler_knn = StandardScaler()
X_trainval_scaled = scaler_knn.fit_transform(np.nan_to_num(X_trainval, nan=0.0))
X_test_scaled = scaler_knn.transform(np.nan_to_num(X_test, nan=0.0))

param_grid = {'n_neighbors': [3, 5, 7, 9, 11], 'weights': ['uniform', 'distance'], 'p': [1, 2]}
grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_trainval_scaled, y_trainval_enc)
best_knn = grid.best_estimator_  # sklearn ga je vec istrenirao na celom X_trainval_scaled
print("Best KNN parameters:", grid.best_params_)

y_test_pred = best_knn.predict(X_test_scaled)
knn_test_acc = accuracy_score(y_test_enc, y_test_pred)
print("Test Accuracy:", knn_test_acc)
print(classification_report(y_test_enc, y_test_pred, target_names=le_knn.classes_))

save_confusion_matrix(y_test_enc, y_test_pred, le_knn.classes_, f"{MODELS_DIR}/knn/test_confusion_matrix.png", "KNN - Test Confusion Matrix")

joblib.dump(best_knn, f"{MODELS_DIR}/knn/knn_model.pkl")
joblib.dump(scaler_knn, f"{MODELS_DIR}/knn/scaler.pkl")
joblib.dump(le_knn, f"{MODELS_DIR}/knn/label_encoder.pkl")


## 6. Trening — Naive Bayes (PCA + GaussianNB)

Isti princip kao kod KNN-a. Dodatno: broj PCA komponenti je ranije bio fiksno postavljen na 50 (nasumična pretpostavka) — sad ga i njega bira `GridSearchCV` (isprobava 20/50/100/150), umesto da ga proizvoljno biramo.

In [ ]:
scaler_nb = StandardScaler()
X_trainval_scaled_nb = scaler_nb.fit_transform(np.nan_to_num(X_trainval, nan=0.0))
X_test_scaled_nb = scaler_nb.transform(np.nan_to_num(X_test, nan=0.0))

nb_pipeline = Pipeline([('pca', PCA()), ('nb', GaussianNB())])
param_grid_nb = {'pca__n_components': [20, 50, 100, 150]}
grid_nb = GridSearchCV(nb_pipeline, param_grid_nb, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_nb.fit(X_trainval_scaled_nb, y_trainval_enc)
nb_final = grid_nb.best_estimator_  # Pipeline (PCA+NB) vec istreniran na celom X_trainval_scaled_nb
print("Best Naive Bayes params (PCA n_components):", grid_nb.best_params_)

y_test_pred_nb = nb_final.predict(X_test_scaled_nb)
nb_test_acc = accuracy_score(y_test_enc, y_test_pred_nb)
print("Test Accuracy:", nb_test_acc)
print(classification_report(y_test_enc, y_test_pred_nb, target_names=le_knn.classes_))

save_confusion_matrix(y_test_enc, y_test_pred_nb, le_knn.classes_, f"{MODELS_DIR}/naive_bayes/test_confusion_matrix.png", "Naive Bayes - Test Confusion Matrix")

joblib.dump(nb_final, f"{MODELS_DIR}/naive_bayes/naive_bayes_model.pkl")
joblib.dump(scaler_nb, f"{MODELS_DIR}/naive_bayes/scaler.pkl")
joblib.dump(le_knn, f"{MODELS_DIR}/naive_bayes/label_encoder.pkl")


## 7. CNN — ekstrakcija Mel spektrograma (sekvence)

Za razliku od KNN/NB, CNN ne koristi statistike (mean/std/max) već **celu vremensku sekvencu** Mel-spektrograma, da bi mreža sama naučila vremenske obrasce.

In [ ]:
sr_cnn = 44100
n_mels_cnn = 128
hop_length = 512
duration_cnn = 3

features_cnn, labels_cnn, actors_cnn = [], [], []
max_frames = 0

for path, label, actor_id in tqdm(zip(df_labels['path'], df_labels['emotion'], df_labels['actor']),
                                   total=len(df_labels), desc="CNN feature ekstrakcija"):
    y, _ = librosa.load(path, sr=sr_cnn, duration=duration_cnn, offset=0.5, res_type='kaiser_fast')
    mel = librosa.feature.melspectrogram(y=y, sr=sr_cnn, n_mels=n_mels_cnn, fmax=8000, hop_length=hop_length)
    mel_db = librosa.power_to_db(mel).T
    features_cnn.append(mel_db.astype(np.float32))
    labels_cnn.append(label)
    actors_cnn.append(actor_id)
    max_frames = max(max_frames, mel_db.shape[0])

features_padded = []
for f in features_cnn:
    if f.shape[0] < max_frames:
        f = np.pad(f, ((0, max_frames - f.shape[0]), (0, 0)), mode='constant')
    else:
        f = f[:max_frames, :]
    features_padded.append(f)

X_all_cnn = np.array(features_padded, dtype=np.float32)
y_all_cnn = np.array(labels_cnn)
actors_all_cnn = np.array(actors_cnn)

train_idx = np.where(~np.isin(actors_all_cnn, TEST_ACTORS + VAL_ACTORS))[0]
val_idx   = np.where(np.isin(actors_all_cnn, VAL_ACTORS))[0]
test_idx  = np.where(np.isin(actors_all_cnn, TEST_ACTORS))[0]

X_train_cnn_raw, y_train_cnn_raw = X_all_cnn[train_idx], y_all_cnn[train_idx]
X_val_cnn,   y_val_cnn   = X_all_cnn[val_idx],   y_all_cnn[val_idx]
X_test_cnn,  y_test_cnn  = X_all_cnn[test_idx],  y_all_cnn[test_idx]

n_mels_dim = X_train_cnn_raw.shape[2]
cnn_scaler = StandardScaler()
cnn_scaler.fit(X_train_cnn_raw.reshape(-1, n_mels_dim))

def apply_cnn_scaler(X):
    shape = X.shape
    return cnn_scaler.transform(X.reshape(-1, n_mels_dim)).reshape(shape)

X_train_cnn_raw = apply_cnn_scaler(X_train_cnn_raw)
X_val_cnn = apply_cnn_scaler(X_val_cnn)
X_test_cnn = apply_cnn_scaler(X_test_cnn)

joblib.dump(cnn_scaler, f"{DATA_PROC}/cnn/cnn_scaler.pkl")
train_paths_df = df_labels.iloc[train_idx][['path', 'emotion']]
train_paths_df.to_csv(f"{DATA_PROC}/cnn/train_cnn_paths.csv", index=False)

target_frames = X_val_cnn.shape[1]
print(f"X_train: {X_train_cnn_raw.shape}, X_val: {X_val_cnn.shape}, X_test: {X_test_cnn.shape}")


### CNN — augmentacija trening skupa

In [ ]:
temp_features, all_emotions_cnn = [], []

for _, row in tqdm(train_paths_df.iterrows(), total=len(train_paths_df), desc="Augmentacija (CNN train)"):
    y, _ = librosa.load(row['path'], sr=sr_cnn, duration=duration_cnn, offset=0.5, res_type='kaiser_fast')
    for func in augment_funcs:
        y_aug = func(y, sr_cnn)
        mel = librosa.feature.melspectrogram(y=y_aug, sr=sr_cnn, n_mels=n_mels_cnn, fmax=8000, hop_length=hop_length)
        mel_db = librosa.power_to_db(mel)
        mel_db = np.nan_to_num(mel_db, nan=0.0, posinf=0.0, neginf=0.0).T
        temp_features.append(mel_db.astype(np.float32))
        all_emotions_cnn.append(row['emotion'])

features_padded_aug = []
for f in temp_features:
    if f.shape[0] < target_frames:
        f = np.pad(f, ((0, target_frames - f.shape[0]), (0, 0)), mode='constant')
    else:
        f = f[:target_frames, :]
    features_padded_aug.append(f)

X_train_cnn = np.array(features_padded_aug, dtype=np.float32)
y_train_cnn = np.array(all_emotions_cnn)

n_samples, n_frames, n_mels_ = X_train_cnn.shape
X_train_cnn = cnn_scaler.transform(X_train_cnn.reshape(-1, n_mels_)).reshape(n_samples, n_frames, n_mels_)

print(f"Augmented CNN train dataset: {X_train_cnn.shape}, {y_train_cnn.shape}")


## 8. Trening — 1D CNN

In [ ]:
from tensorflow.keras.layers import GlobalAveragePooling1D, Conv1D, MaxPooling1D, Dropout, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

le_cnn = LabelEncoder()
le_cnn.fit(EMOTIONS)
y_train_cnn_int = le_cnn.transform(y_train_cnn)
y_val_cnn_int   = le_cnn.transform(y_val_cnn)
y_test_cnn_int  = le_cnn.transform(y_test_cnn)

num_classes = len(le_cnn.classes_)
y_train_cnn_enc = to_categorical(y_train_cnn_int, num_classes=num_classes)
y_val_cnn_enc   = to_categorical(y_val_cnn_int, num_classes=num_classes)
y_test_cnn_enc  = to_categorical(y_test_cnn_int, num_classes=num_classes)

X_train_cnn_in = X_train_cnn[:, :, np.newaxis] if X_train_cnn.ndim == 2 else X_train_cnn
X_val_cnn_in   = X_val_cnn[:, :, np.newaxis] if X_val_cnn.ndim == 2 else X_val_cnn
X_test_cnn_in  = X_test_cnn[:, :, np.newaxis] if X_test_cnn.ndim == 2 else X_test_cnn

input_shape = X_train_cnn_in.shape[1:]

model = Sequential([
    Conv1D(64, kernel_size=20, activation='relu', input_shape=input_shape),
    Conv1D(128, kernel_size=20, activation='relu', kernel_regularizer=l2(0.01), bias_regularizer=l2(0.01)),
    MaxPooling1D(pool_size=8),
    Dropout(0.4),
    Conv1D(128, kernel_size=20, activation='relu'),
    MaxPooling1D(pool_size=8),
    Dropout(0.4),
    GlobalAveragePooling1D(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(num_classes, activation='softmax'),
])

model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.0001), metrics=['accuracy'])
model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ModelCheckpoint(f"{MODELS_DIR}/cnn/cnn_model.h5", monitor='val_loss', save_best_only=True),
]

history = model.fit(
    X_train_cnn_in, y_train_cnn_enc,
    validation_data=(X_val_cnn_in, y_val_cnn_enc),
    epochs=50, batch_size=64, callbacks=callbacks,
)
model.save(f"{MODELS_DIR}/cnn/cnn_model.h5")


In [ ]:
cnn_test_loss, cnn_test_acc = model.evaluate(X_test_cnn_in, y_test_cnn_enc)
print(f"CNN Test accuracy: {cnn_test_acc*100:.2f}%")

y_pred_probs = model.predict(X_test_cnn_in)
y_pred_cnn = np.argmax(y_pred_probs, axis=1)
y_true_cnn = np.argmax(y_test_cnn_enc, axis=1)
print(classification_report(y_true_cnn, y_pred_cnn, target_names=le_cnn.classes_))

save_confusion_matrix(y_true_cnn, y_pred_cnn, le_cnn.classes_, f"{MODELS_DIR}/cnn/confusion_matrix.png", "CNN - Test Confusion Matrix")

train_loss = history.history['loss']; val_loss = history.history['val_loss']
train_acc = history.history['accuracy']; val_acc = history.history['val_accuracy']
epochs_range = range(1, len(train_loss) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_loss, 'b', label='Train loss')
plt.plot(epochs_range, val_loss, 'r', label='Val loss')
plt.legend(); plt.title('Loss')
plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_acc, 'b', label='Train acc')
plt.plot(epochs_range, val_acc, 'r', label='Val acc')
plt.legend(); plt.title('Accuracy')
plt.savefig(f"{MODELS_DIR}/cnn/training_curves.png")
plt.show()


## 9. Poređenje KNN / Naive Bayes / CNN

Ovo je poslednji korak koji ti treba **odmah** — daje ti gotovu tabelu i grafik za sva tri postojeća modela. Wav2Vec2 (10. sekcija ispod) je poseban, nezavisan dodatak koji možeš pokrenuti bilo kada kasnije — ne mora danas.

In [ ]:
results = pd.DataFrame({
    "model": ["KNN", "Naive Bayes", "CNN"],
    "test_accuracy": [knn_test_acc, nb_test_acc, cnn_test_acc],
}).sort_values("test_accuracy", ascending=False).reset_index(drop=True)

results.to_csv(f"{MODELS_DIR}/comparison.csv", index=False)
display(results)

plt.figure(figsize=(7, 5))
sns.barplot(data=results, x="model", y="test_accuracy")
plt.ylim(0, 1)
plt.ylabel("Test accuracy")
plt.title("Poredjenje modela - actor-independent test skup")
for i, v in enumerate(results["test_accuracy"]):
    plt.text(i, v + 0.01, f"{v*100:.1f}%", ha="center")
plt.savefig(f"{MODELS_DIR}/comparison.png")
plt.show()


### Napomena za diplomski rad

Ako su ovi rezultati niži od ranije prijavljenih (npr. stari CNN rezultat od 81%), to je **očekivano i treba tako obrazložiti u radu**: ranija evaluacija je koristila nasumičnu (stratifikovanu) podelu na train/val/test, gde se isti glumac mogao naći i u train i u test skupu — model je delom prepoznavao boju glasa konkretne osobe, a ne samu emociju, pa je izmerena tačnost bila veštački visoka. Ova sveska koristi **actor-independent** podelu, koja realno meri sposobnost generalizacije modela na *nove, neviđene govornike* — što je i realan cilj sistema za prepoznavanje emocija iz govora.

### Preuzimanje rezultata (VAŽNO!)

Colab briše sve fajlove kad se sesija ugasi ili zatvoriš tab. Pokreni ćeliju ispod da spakuješ sve modele, slike i CSV-ove u jedan `.zip` i preuzmeš ga na svoj računar.

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/emotion_detector_results"
shutil.make_archive(zip_path, "zip", MODELS_DIR)
files.download(zip_path + ".zip")


## 10. (Kasnije) NOVI, četvrti MODEL — Wav2Vec2 (transfer learning / fine-tuning)

**Ovu sekciju ne moraš da radiš danas.** Kad budeš spreman/na: samo ponovo pokreni ceo notebook (Runtime → Run all) do ovde (sekcije 1-9 moraju biti izvršene u istoj sesiji jer ova sekcija koristi `df_labels`, `TRAIN_ACTORS` itd. iz ranijih ćelija), pa nastavi odavde nadole.

Umesto ručno dizajniranih Mel karakteristika, ovde koristimo **Wav2Vec2** (`facebook/wav2vec2-base`) — transformer model koji je *self-supervised* pretreniran na ogromnoj količini govora (Librispeech). Fine-tune-ujemo ga direktno na sirovom audio signalu (bez mel-spektrograma) za klasifikaciju emocija.

**Napomena za rad (metodologija):** ovo je transfer learning, ne trening od nule — model već "zna" opšte akustičke osobine govora sa Librispeech-a, mi ga samo doučavamo (fine-tune-ujemo) za prepoznavanje emocija na RAVDESS-u. Donji, konvolucioni deo mreže (feature encoder) je zamrznut (već dobro naučen), fine-tune-uje se samo transformer + klasifikaciona glava. Koristi **istu actor-independent podelu** kao ostali modeli — pa je poređenje pošteno.

In [ ]:
# NE pinujemo transformers na staru verziju - Colab vec ima instaliranu novu (5.x),
# a pokusaj instalacije stare verzije puca jer stari "tokenizers" (Rust ekstenzija)
# ne moze da se izgradi na novijem Python-u.
# NE instaliramo "datasets" ni "evaluate" - ne koristimo ih nigde u kodu (imamo
# sopstveni PyTorch Dataset i koristimo sklearn za metrike), a njihova instalacija
# zna da povuce noviji "pyarrow" koji se kosi sa onim sto Colab vec ima ucitano
# (ValueError: pyarrow... binary incompatibility). Instaliramo SAMO accelerate,
# koji je Trainer-u stvarno potreban, bez -U (da ne diramo nista sto vec radi).
!pip install -q accelerate

import transformers
print("Aktivna verzija transformers:", transformers.__version__)


In [ ]:
import torch
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

WAV2VEC2_CHECKPOINT = "facebook/wav2vec2-base"
TARGET_SR = 16000
CLIP_DURATION = 3  # sekunde, isto kao ostatak pipeline-a

le_w2v = LabelEncoder()
le_w2v.fit(EMOTIONS)

train_paths_w2v = df_labels[df_labels['actor'].isin(TRAIN_ACTORS)].reset_index(drop=True)
val_paths_w2v   = df_labels[df_labels['actor'].isin(VAL_ACTORS)].reset_index(drop=True)
test_paths_w2v  = df_labels[df_labels['actor'].isin(TEST_ACTORS)].reset_index(drop=True)

print(f"Train: {len(train_paths_w2v)}, Val: {len(val_paths_w2v)}, Test: {len(test_paths_w2v)}")


In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAV2VEC2_CHECKPOINT)

# Ove funkcije vec postoje u sekciji 4 (KNN/NB), ali ih ovde definisemo ponovo
# da ova sekcija (10) bude potpuno samostalna - da moze da se pokrene i ako si
# preskocila sekcije 4-9 (samo su ti trebale sekcije 1-3 pre ovoga).
def add_noise(data, noise_factor=0.005):
    return data + noise_factor * np.random.randn(len(data))

def time_shift(data, shift_max=0.2):
    shift = np.random.randint(int(len(data) * -shift_max), int(len(data) * shift_max))
    return np.roll(data, shift)

class RavdessAudioDataset(torch.utils.data.Dataset):
    """Ucitava sirovi audio na 16kHz, sece/dopunjava na fiksnu duzinu za Wav2Vec2."""
    def __init__(self, df, label_encoder, augment=False):
        self.paths = df['path'].tolist()
        self.labels = label_encoder.transform(df['emotion'].tolist())
        self.augment = augment
        self.target_len = TARGET_SR * CLIP_DURATION

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        y, _ = librosa.load(self.paths[idx], sr=TARGET_SR, duration=CLIP_DURATION, offset=0.5)
        if self.augment:
            choice = np.random.choice(["none", "noise", "shift"])
            if choice == "noise":
                y = add_noise(y)
            elif choice == "shift":
                y = time_shift(y)
        if len(y) < self.target_len:
            y = np.pad(y, (0, self.target_len - len(y)))
        else:
            y = y[:self.target_len]
        inputs = feature_extractor(y, sampling_rate=TARGET_SR, return_tensors="np")
        return {
            "input_values": inputs["input_values"][0].astype(np.float32),
            "labels": int(self.labels[idx]),
        }

train_dataset_w2v = RavdessAudioDataset(train_paths_w2v, le_w2v, augment=True)
val_dataset_w2v   = RavdessAudioDataset(val_paths_w2v, le_w2v, augment=False)
test_dataset_w2v  = RavdessAudioDataset(test_paths_w2v, le_w2v, augment=False)


In [ ]:
w2v_model = Wav2Vec2ForSequenceClassification.from_pretrained(
    WAV2VEC2_CHECKPOINT,
    num_labels=len(le_w2v.classes_),
    label2id={l: i for i, l in enumerate(le_w2v.classes_)},
    id2label={i: l for i, l in enumerate(le_w2v.classes_)},
)
# Zamrzavamo konvolucioni feature-encoder sloj (nizak nivo audio karakteristika,
# vec dobro naucen na Librispeech-u) - fine-tune-ujemo samo transformer + klasifikacionu glavu.
# Ovo znatno smanjuje rizik od overfitting-a na malom RAVDESS skupu.
w2v_model.freeze_feature_encoder()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

# warmup_steps umesto warmup_ratio: warmup_ratio ne postoji u svim verzijama
# transformers-a; warmup_steps postoji u bas svakoj, pa ga racunamo rucno
# (10% od ukupnog broja koraka treninga).
BATCH_SIZE = 4
GRAD_ACCUM = 4
NUM_EPOCHS = 10
steps_per_epoch = max(1, len(train_dataset_w2v) // (BATCH_SIZE * GRAD_ACCUM))
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = max(1, int(0.1 * total_steps))

# Naziv parametra za "kad da se evaluira" se menjao kroz verzije transformers-a
# (evaluation_strategy -> eval_strategy). Umesto da pogadjamo, proveravamo koji
# naziv TRENUTNO instalirana verzija stvarno prihvata.
import inspect
ta_params = inspect.signature(TrainingArguments.__init__).parameters
eval_strategy_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"

training_args = TrainingArguments(
    output_dir=f"{MODELS_DIR}/wav2vec2/checkpoints",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=3e-5,
    warmup_steps=warmup_steps,
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    report_to="none",
    **{eval_strategy_key: "epoch"},
)

trainer = Trainer(
    model=w2v_model,
    args=training_args,
    train_dataset=train_dataset_w2v,
    eval_dataset=val_dataset_w2v,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
test_results = trainer.predict(test_dataset_w2v)
y_pred_w2v = np.argmax(test_results.predictions, axis=1)
y_true_w2v = test_results.label_ids

w2v_test_acc = accuracy_score(y_true_w2v, y_pred_w2v)
print(f"Wav2Vec2 Test accuracy: {w2v_test_acc*100:.2f}%")
print(classification_report(y_true_w2v, y_pred_w2v, target_names=le_w2v.classes_))

save_confusion_matrix(y_true_w2v, y_pred_w2v, le_w2v.classes_, f"{MODELS_DIR}/wav2vec2/test_confusion_matrix.png", "Wav2Vec2 - Test Confusion Matrix")

trainer.save_model(f"{MODELS_DIR}/wav2vec2/final_model")
feature_extractor.save_pretrained(f"{MODELS_DIR}/wav2vec2/final_model")
joblib.dump(le_w2v, f"{MODELS_DIR}/wav2vec2/label_encoder.pkl")


## 11. Ažurirano poređenje (sa Wav2Vec2)

In [ ]:
results = pd.DataFrame({
    "model": ["KNN", "Naive Bayes", "CNN", "Wav2Vec2"],
    "test_accuracy": [knn_test_acc, nb_test_acc, cnn_test_acc, w2v_test_acc],
}).sort_values("test_accuracy", ascending=False).reset_index(drop=True)

results.to_csv(f"{MODELS_DIR}/comparison.csv", index=False)
display(results)

plt.figure(figsize=(7, 5))
sns.barplot(data=results, x="model", y="test_accuracy")
plt.ylim(0, 1)
plt.ylabel("Test accuracy")
plt.title("Poredjenje modela - actor-independent test skup")
for i, v in enumerate(results["test_accuracy"]):
    plt.text(i, v + 0.01, f"{v*100:.1f}%", ha="center")
plt.savefig(f"{MODELS_DIR}/comparison.png")
plt.show()


### Preuzimanje ažuriranih rezultata (sa Wav2Vec2)

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/emotion_detector_results_final"
shutil.make_archive(zip_path, "zip", MODELS_DIR)
files.download(zip_path + ".zip")
